In [ ]:
import folium
import geopandas as gpd
from IPython.display import HTML

# 1. Cargar archivo local
colonias_cdmx = gpd.read_file("../data/mapa_colonias.json").to_crs(epsg=4326)

# 2. Localizar columna de nombre de colonia
col_nom = next(
    c
    for c in colonias_cdmx.columns
    if any(k in c.lower() for k in ["colonia", "nombre", "nom"])
)

# 3. Filtrar por las 3 alcaldías
# Revisamos si existe una columna de texto ('alc', 'alcaldia') o si usamos la clave 'cve_alc'
# Claves oficiales CDMX: Milpa Alta (9 o 009), Tláhuac (11 o 011), Xochimilco (13 o 013)
CLAVES_SUR = ["009", "011", "013", "9", "11", "13", "09", "11", "13"]

# Buscamos si hay columna con texto de alcaldía
cols_texto_alc = [
    c
    for c in colonias_cdmx.columns
    if any(k in c.lower() for k in ["alcaldia", "alc"]) and c != "cve_alc"
]

if cols_texto_alc:
    col_alc = cols_texto_alc[0]
    filtro_sur = colonias_cdmx[col_alc].astype(str).str.upper().str.contains(
        "XOCHIMILCO|TL[AÁ]HUAC|MILPA ALTA", regex=True
    )
else:
    # Usamos cve_alc convirtiendo a string sin ceros o con ceros a la izquierda
    col_alc = "cve_alc"
    # Normalizamos el string quitando ceros iniciales para asegurar coincidencia
    cves_limpias = colonias_cdmx["cve_alc"].astype(str).str.lstrip("0")
    filtro_sur = cves_limpias.isin(["9", "11", "13"])

colonias_sur = colonias_cdmx[filtro_sur].copy()
print(f"Total de polígonos cargados: {len(colonias_cdmx)}")
print(f"Columna utilizada para alcaldía: '{col_alc}'")
print(f"Unidades territoriales en el suroriente encontradas: {len(colonias_sur)}")

# 4. Renderizar con .explore() sin conflicto de tipos en tooltip
# Seleccionamos solo las columnas de interés para que el tooltip las muestre limpias
cols_vis = [col_nom, col_alc, "geometry"]
m = colonias_sur[cols_vis].explore(
    column=col_alc,
    tooltip=True,  # Muestra todas las columnas del dataframe recortado en el tooltip
    legend=True,
    style_kwds={"fillOpacity": 0.45, "weight": 1.2},
)

HTML(m._repr_html_())

## Delimitacion de alcaldias

In [ ]:
import geopandas as gpd
from IPython.display import HTML

# 1. Claves oficiales de alcaldías del suroriente
# 09009: Milpa Alta | 09011: Tláhuac | 09013: Xochimilco
MUNICIPIOS_SURORIENTE = ["009", "011", "013"]

# 2. Cargar capas estatales completas y proyectar a WGS84 (EPSG:4326)
# Ajusta el nombre de archivo según como se llamen tus .shp de CDMX
urbano_cdmx = gpd.read_file("../data/ageb_urbano/ageb_urbano.shp").to_crs(epsg=4326)
rural_cdmx = gpd.read_file("../data/ageb_rural/ageb_rural.shp").to_crs(epsg=4326)

# 3. Filtrar estrictamente por las 3 alcaldías
urbano_sur = urbano_cdmx[
    urbano_cdmx["CVE_MUN"].isin(MUNICIPIOS_SURORIENTE)
].copy()
rural_sur = rural_cdmx[rural_cdmx["CVE_MUN"].isin(MUNICIPIOS_SURORIENTE)].copy()

# Mapear nombre de alcaldía para facilitar lectura en mapa/tablas
nombres_alc = {"009": "Milpa Alta", "011": "Tláhuac", "013": "Xochimilco"}
urbano_sur["ALCALDIA"] = urbano_sur["CVE_MUN"].map(nombres_alc)
rural_sur["ALCALDIA"] = rural_sur["CVE_MUN"].map(nombres_alc)

print(f"AGEBs urbanas encontradas: {len(urbano_sur)}")
print(f"AGEBs rurales encontradas: {len(rural_sur)}")

In [ ]:
import folium

# Centrado geográfico entre Xochimilco, Tláhuac y Milpa Alta (aprox. faldas del Teuhtli)
m = folium.Map(
    location=[19.220, -99.030], zoom_start=11, tiles="CartoDB positron"
)

# Capa Satelital ESRI
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="ESRI Satellite",
    name="Satélite ESRI",
).add_to(m)

# 1. Capa Urbana (Rojo / Naranja)
folium.GeoJson(
    urbano_sur,
    name="Área Urbana (AGEB Urbano)",
    style_function=lambda x: {
        "fillColor": "#e41a1c",
        "color": "#b10026",
        "weight": 1,
        "fillOpacity": 0.45,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["ALCALDIA", "CVE_AGEB"], aliases=["Alcaldía:", "AGEB:"]
    ),
).add_to(m)

# 2. Capa Rural (Verde / Suelo de Conservación)
folium.GeoJson(
    rural_sur,
    name="Área Rural (AGEB Rural)",
    style_function=lambda x: {
        "fillColor": "#33a02c",
        "color": "#006837",
        "weight": 1.5,
        "fillOpacity": 0.35,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["ALCALDIA", "CVE_AGEB"], aliases=["Alcaldía:", "AGEB:"]
    ),
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# Render en VS Code
HTML(m._repr_html_())